In [119]:
import pandas as pd
import numpy as np
import re

Load the raw data

In [120]:
# Load the data
df = pd.read_csv("../data/raw/indigenous-business/bcindigenousbusinesslistings.csv")

Inspecting the data

In [121]:
# Inspect the data
print(df.info())
print(df.head())
print(f"Initial number of rows: {len(df)}") 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Business Name        1259 non-null   object 
 1   Description          1135 non-null   object 
 2   Web Site             699 non-null    object 
 3   City                 1258 non-null   object 
 4   Latitude             1258 non-null   float64
 5   Longitude            1258 non-null   float64
 6   Keywords             1257 non-null   object 
 7   Region               1259 non-null   object 
 8   Type                 1123 non-null   object 
 9   Industry Sector      1222 non-null   object 
 10  Year Formed          648 non-null    float64
 11  Number of Employees  572 non-null    object 
dtypes: float64(3), object(9)
memory usage: 118.2+ KB
None
                                       Business Name  \
0                                Ellipsis Energy Inc   
1  Indigenous Communit

Column Name Standardization

In [122]:
#  Clean column names (convert to lowercase and replace spaces with underscores)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

Remove Unnecessary Columns

In [123]:
# Remove unnecessary columns
columns_to_drop = ['description', 'web_site', 'keywords']
df = df.drop(columns=columns_to_drop, errors='ignore')

Removal of Duplicates

In [124]:
# Remove duplicate rows
df = df.drop_duplicates()

In [125]:
# check no of rows after removing duplicates
print(f"No of rows after removing duplicates: {len(df)}") 

No of rows after removing duplicates: 1259


Critical Data Validation

In [126]:
# Remove rows missing critical information
#business_name is a mandatory field here
if 'business_name' in df.columns:
    df = df[df['business_name'].notna() & (df['business_name'] != '')]

In [127]:
# check no of rows after removing rows missing critical information
print(f"No of rows: {len(df)}") 

No of rows: 1259


Ensure Year is an integer

In [128]:
# Ensure year_formed is a nullable integer
df['year_formed'] = pd.to_numeric(df['year_formed'], errors='coerce').astype('Int64')

Cleanup industry_sector

In [129]:
# custom function to standardize industry_sector data
def clean_industry_sector(sector):
    if pd.isna(sector):
        return np.nan
    
    # Convert to string
    sector = str(sector).strip()
    
    # Handle cases starting with colon
    if sector.startswith(':'):
        sector = sector[1:].strip()
    
    # Remove ALL number patterns including:
    # "23 - ", "44-45 - ", "1.5 - ", "54 – " (with en dash)
    sector = re.sub(r'^[\d\.]+\s*[-–—]?\s*[\d\.]*\s*[-–—]\s*', '', sector).strip()
    
    # Return np.nan if empty, otherwise capitalize first letter
    return np.nan if not sector else sector[0].upper() + sector[1:]


print("test cleaning:")
test_case = ":54 – Professional, scientific and technical services"
print(f"'{test_case}' → '{clean_industry_sector(test_case)}'")


df['industry_sector'] = df['industry_sector'].apply(clean_industry_sector)

test cleaning:
':54 – Professional, scientific and technical services' → 'Professional, scientific and technical services'


Data Formatting

In [130]:
# Trim whitespace in string fields
text_cols = ['business_name', 'city', 'industry_sector','region','type']
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())

Clean the Region Categories (Combine "Vancouver Island And Coast" and "Vancouver Island / Coast" to "Vancouver Island / Coast")

In [131]:
# check current region column categories
df['region'].value_counts()

region
Lower Mainland / Southwest    343
Vancouver Island / Coast      278
Thompson / Okanagan           193
North Coast                   169
Northeast                      86
Nechako                        76
Cariboo                        58
Kootenay                       41
Vancouver Island and Coast     15
Name: count, dtype: int64

In [132]:
# replace 'Vancouver Island and Coast' with 'Vancouver Island / Coast'
df.loc[df['region'] == 'Vancouver Island and Coast', 'region'] = 'Vancouver Island / Coast'

In [133]:
# check region column categories after value replacement
df['region'].value_counts()

region
Lower Mainland / Southwest    343
Vancouver Island / Coast      293
Thompson / Okanagan           193
North Coast                   169
Northeast                      86
Nechako                        76
Cariboo                        58
Kootenay                       41
Name: count, dtype: int64

Clean Business Ownership Types

In [134]:
df.columns.to_list()

['business_name',
 'city',
 'latitude',
 'longitude',
 'region',
 'type',
 'industry_sector',
 'year_formed',
 'number_of_employees']

In [135]:
import string

In [136]:
df['city'] = df['city'].str.replace('.', '')
df['city'] = df['city'].str.replace(',', '')
df['city'] = df['city'].str.replace('BC', '')
df['city'] = df['city'].apply(
    lambda x: string.capwords(x) if isinstance(x, str) else x
)
df["city"] = df["city"].apply(
    lambda x: re.sub(r"\s+", " ", x) if isinstance(x, str) else x
)
df['city'] = df['city'].str.strip()

In [137]:
city_counts_df = pd.DataFrame(df['city'].value_counts()).reset_index().sort_values(by='city')
city_counts_df

,city,count
103,100 Mile House,2
123,150 Mile House,2
167,Abbotsfford,1
26,Abbotsford,12
4,Agassiz,31
...,...,...
151,Witset,1
33,Wonowon,11
175,Wonowoon,1
161,Yale,1


In [138]:
cities_list = list(city_counts_df['city'].unique())

import difflib

matches = []

for city in cities_list:
    match = difflib.get_close_matches(
        city,
        cities_list,
        cutoff=0.75
    )
    
    match = sorted(match)

    if len(match) > 1 and match not in matches:
        matches.append(match)

matches

[['100 Mile House', '150 Mile House'],
 ['Abbotsfford', 'Abbotsford'],
 ['Brurns Lake', 'Burns Lake'],
 ['Cambell River', 'Campbell River'],
 ['Christina Lake', 'Nitinat Lake'],
 ['Coquitlam', 'Cquitlam', 'Port Coquitlam'],
 ['Cowichan', 'Cowichan Bay'],
 ['Coquitlam', 'Cquitlam'],
 ['Dawson Creek', 'Dog Creek'],
 ['Dease Lake', 'Fraser Lake'],
 ['Fort St John', 'Ft St John'],
 ['Hazelton', 'New Hazelton', 'Old Hazelton'],
 ['Lilloet', 'Lillooet', 'Lilooet'],
 ['Lytton', 'Lyyton'],
 ['Masset', 'Massett', 'Old Masset'],
 ['Masset', 'Massett'],
 ['Merrit', 'Merritt'],
 ['North Vancouver', 'Vancouver', 'West Vancouver'],
 ['Masset', 'Old Masset'],
 ['Penticon', 'Penticton'],
 ['Coquitlam', 'Port Coquitlam'],
 ['Port Edward', 'Port Hardy'],
 ['Shalalth', 'Shalath'],
 ['Telegraph Cove', 'Telegraph Creek'],
 ['Ucluelet', 'Uculet'],
 ['West Bank', 'Westbank'],
 ['Wonowon', 'Wonowoon']]

In [ ]:
city_counts_df = pd.DataFrame(df['city'].value_counts()).reset_index().sort_values(by='city')
city_counts_df_under_5 = city_counts_df[city_counts_df['count'] == 1]


"                    city  count\n176          Abbotsfford      1\n138          Alkali Lake      1\n219               Anmore      1\n153            Baldonnel      1\n152          Black Pines      1\n180         Bowen Island      1\n215        Brentwood Bay      1\n186          Brurns Lake      1\n174                Buick      1\n159        Cambell River      1\n133               Canyon      1\n211           Cape Mudge      1\n233            Castlegar      1\n228       Chilanko Forks      1\n168       Christina Lake      1\n150           Clearwater      1\n206         Coal Harbour      1\n191          Cobble Hill      1\n157             Cowichan      1\n143         Cowichan Bay      1\n205             Cquitlam      1\n236          Cultus Lake      1\n210               D'Arcy      1\n218           DEASE LAKE      1\n230         Douglas Lake      1\n199             Falkland      1\n148         Forest Grove      1\n197       Fort  St James      1\n145    Fort St John B.C.      1\n134      

In [ ]:
def normalize(string):
    string = string.strip()
    string = string.replace(".", "")
    string = string.replace(",", "")
    string = string.replace(".", "")

    return string

Save the Cleaned data

In [16]:
# Save cleaned data
df.to_csv("../data/clean/bcindigenousbiz.csv", index=False)

Validation of Cleaned Data

In [17]:
# Validate cleaned data
clean_data= pd.read_csv("../data/clean/bcindigenousbiz.csv")
print(f"Final cleaned dataset rows: {len(clean_data)}")  # Final row count
clean_data.head()

Final cleaned dataset rows: 1259


,business_name,city,latitude,longitude,region,type,industry_sector,year_formed,number_of_employees
0,Ellipsis Energy Inc,Moberly Lake,55.819370,-121.834602,Northeast,Private Company,"Mining, quarrying, and oil and gas extraction",2012.0,5 to 9
1,Indigenous Community Development & Prosperity ...,Enderby,50.551498,-119.133546,Thompson / Okanagan,Private Company,Other services (except public administration),2020.0,1 to 4
2,Formline Construction Ltd.,Burnaby,49.266050,123.005840,Lower Mainland / Southwest,Private Company,Construction,2021.0,1 to 4
3,Quilakwa Investments Ltd.,Enderby,50.537507,-119.141955,Thompson / Okanagan,Community Owned Company,Accommodation and food services,1984.0,20 to 49
4,Quilakwa Esso,Enderby,50.537507,-119.141955,Thompson / Okanagan,Community Owned Company,Retail trade,1984.0,10 to 19
